Exploration of the new data from the new CAF scraping module thing that Stoyan made to get C objects into a readable state for python tools

In [ ]:
import sys
import uproot
import pandas as pd
import numpy as np
sys.path.append("/users/oz22897/atmospherics-tools/fast-osc-feedback")

from fastfeedback import *

In [ ]:


file_path = "reco_pfp_info.root"
tree = uproot.open(file_path)["reco_pfp_tree"]

df = tree.arrays(library="pd")

print(f"entries = {len(df)}")
df.head()

entries = 2884703


,reco_ok,n_reco_pfps,n_reco_tracks,n_reco_showers,single_hits_energy,n_reco_muons_pions,n_reco_protons
0,True,10,5,4,11.48799,4,1
1,True,2,1,0,0.11094,1,0
2,True,4,3,0,0.06457,2,1
3,False,-1,-1,-1,-1.00000,-1,-1
4,True,3,2,0,0.00242,1,1


In [30]:
import uproot
import pandas as pd
import numpy as np

MAIN_CAF = "/storage/1/st15719/caf_new_sum.2.6M_weighted.root"
PFP_FILE = "reco_pfp_info.root"

P_ENU = "rec/common/common.ixn.pandora/common.ixn.pandora.Enu.calo"
P_ELEP = "rec/common/common.ixn.pandora/common.ixn.pandora.Enu.lep_calo"

caf_tree = uproot.open(MAIN_CAF)["cafTree"]
data = caf_tree.arrays([P_ENU, P_ELEP])

initial_count = len(data[P_ENU])
print(f"Initial entries in CAF file: {initial_count}")

df_kinematics = pd.DataFrame({
    'E_nu': [x[0] if len(x) > 0 else np.nan for x in data[P_ENU]],
    'E_lep': [x[0] if len(x) > 0 else np.nan for x in data[P_ELEP]]
})

df_kinematics['hadron_energy'] = df_kinematics['E_nu'] - df_kinematics['E_lep']
df_kinematics['inelasticity'] = (df_kinematics['E_nu'] - df_kinematics['E_lep']) / df_kinematics['E_nu']

pfp_tree = uproot.open(PFP_FILE)["reco_pfp_tree"]
df_pfps = pfp_tree.arrays(library="pd")

df_final = pd.concat([df_kinematics, df_pfps], axis=1)

count_pre_filter = len(df_final)
df_final = df_final[df_final['E_nu'] > -998]
count_after_enu = len(df_final)
print(f"Entries discarded by E_nu > -998 filter: {count_pre_filter - count_after_enu}")

df_final = df_final.dropna()
count_after_dropna = len(df_final)
print(f"Entries discarded by dropna(): {count_after_enu - count_after_dropna}")

print(f"\nFinal DataFrame contains {len(df_final)} events.")
print(df_final[['E_nu', 'hadron_energy', 'inelasticity', 'n_reco_pfps']].head())

Initial entries in CAF file: 2884703
Entries discarded by E_nu > -998 filter: 233676
Entries discarded by dropna(): 0

Final DataFrame contains 2651027 events.
        E_nu  hadron_energy  inelasticity  n_reco_pfps
0  32.386795       3.856979      0.119091           10
1   0.215232      -0.107024     -0.497252            2
2   1.178813      -0.022898     -0.019424            4
4   0.507301      -0.086399     -0.170311            3
5   0.301862      -0.612872     -2.030306            2
